#### validation_experiments.ipynb
This notebook provides a series of simple demonstrations of the fundamental idea of using local sea-level fluctuations to discover TPW signals, with the use of synthetic test data.

In [ ]:
import numpy as np
import pandas as pd
import pygplates as pygp
import matplotlib.pyplot as plt
import methods as m

In [ ]:
# First prescribe a TPW rotation pole (anti-clockwise sense) to see the expected global sea-level change
lam = 60                            # location of TPW axis (clockwise rotation)
fig = m.plot_Y21(lam, levels=31)    # plot Y21 (degree 2, order 1 spherical harmonic) signal given TPW axis location
plt.show()

In [ ]:
# Now we see if we can recover this signal from a set of 'observations'. We start by generating some data from this Y21 signal at some specified spatial resolution
resolution = 3                                  # H3 resolution
hexagons, lats, lons = m.make_grid(resolution)  # make reference grid
s = m.Y21(lam, lats, lons)                      # generate Y21 response at grid points, given the TPW axis location
s_noisy = m.add_noise(s, sigma=0.3)             # add noise to simulated data
fig = m.plot_flux(lam, lats, lons, s_noisy)     # plot predicted sea-level variations
plt.show()

In [ ]:
# Now 'flatten' the sea-level variations to mimic the kinds of binary observations that we typically have (regression / transgression)
ts, tlats, tlons = m.remove_zeros(m.make_ternary(s_noisy, -0.5, 0.5), lats, lons)   # flatten to -1 (regression), 0 (no change), or 1 (transgression) and discard the zeros.
fig = m.plot_flux(lam, tlats, tlons, ts)
plt.show()

In [ ]:
# Let's see if we can recover the known TPW axis location using the flattened observations
y = (ts == 1).astype(int)                 # convert -1/1 data to 0/1 for logistic regression
df = m.invert_lam_logistic(tlats, tlons, y)        # conduct inversion by checking each possible TPW axis location (0 to 360 degrees)
fig = m.plot_inversion(df)                # plot the model coefficients against TPW axis location 
plt.show()

In [ ]:
# We can also inspect the logistic regression of the best-fitting model (or any of the other models)
mask = df['beta'] > 0
best_idx = df.loc[mask, 'llr'].idxmax()                  # isolate the best fitting model
sbest = m.Y21(df.loc[best_idx, 'lambda'], tlats, tlons)  # get predictor values for best-fitting model
m.plot_log_reg(sbest, y)                                 # plot logistic regression
plt.show()

In [ ]:
# Another thing we may consider is how much resolving power our network of observations have, and how isotropic the collection is
rlats, rlons = m.equisphere(len(ts))            # take the same number of observations but randomly distributed on sphere
revals, revecs = m.eigendecomp(rlats, rlons)    # compute the eigenvalues & eigenvectors describing this distribution (relative to Y21)
ref_power = np.sum(revals)                      # resolving power (of random distribution with same number of observations)

evals, evecs = m.eigendecomp(tlats, tlons)      # repeat on actual observations
aniso = evals[0]/evals[1]                       # anisotropy is the ratio of the eigenvalues
m.plot_resolving_pwr(evals, evecs, ref_power)   # show the anisotropy relative to unit-circle 
plt.show()

In [ ]:
# Now let's consider a case where there is an additional eustatic signal (a global bias of constant sign)
s_biased = s + 0.33                                     # add eustatic bias
s_biased_noisy = m.add_noise(s_biased, sigma=0.3)       # add noise again
fig = m.plot_flux(lam, lats, lons, s_biased_noisy)
plt.show()

In [ ]:
ts, tlats, tlons = m.remove_zeros(m.make_ternary(s_biased_noisy, -0.5, 0.5), lats, lons)  # Flatten again
fig = m.plot_flux(lam, tlats, tlons, ts)
plt.show()

In [ ]:
# And run inversion
y = (ts == 1).astype(int)
df = m.invert_lam_logistic(tlats, tlons, y)
fig = m.plot_inversion(df)
plt.show()

In [ ]:
# Note that the decision boundary is now shifted in accordance with the shifted alpha coefficient seen above (reflecting the presence of a eustatic bias)
mask = df['beta'] > 0
best_idx = df.loc[mask, 'llr'].idxmax()                  
sbest = m.Y21(df.loc[best_idx, 'lambda'], tlats, tlons)  
m.plot_log_reg(sbest, y)                                 
plt.show()

In [ ]:
# And the observational network has become more isotropic because the eustatic contribution has acted to suppress the nodal lines
evals, evecs = m.eigendecomp(tlats, tlons)      
aniso = evals[0]/evals[1]                       
m.plot_resolving_pwr(evals, evecs, ref_power) 
plt.show()

In [ ]:
# Now we consider what happens with an irregular and heavily restricted spatial distribution of observations. 
# For this we'll use a real observation distribution from some specified time

lam = 30    # re-select TPW axis location (if desired)
t = 250     # select a time-frame for which to collect observation points

polygons = pygp.FeatureCollection(f'plate_model/static_polygons.gpml')       # continental polygons
rot_model = f'plate_model/PHAB25_zero-lon-Africa.rot'                          # gplates rotation file
anchor_plate = 998                                                           # anchor plate ID
recon_polys = m.reconstruct_polygons(polygons, rot_model, t, anchor_plate)   # reconstruct polygons

t0 = pd.read_csv(f'obs_grids/{t}_Ma.csv')                # read in observation grids
t1 = pd.read_csv(f'obs_grids/{t+10}_Ma.csv')
merged = pd.merge(t0, t1, on='name', suffixes=('_t0', '_t1'))                # merge and keep only points where there is a change between maps
merged['diff'] =  merged['state_t1'] - merged['state_t0']
fluxpts = merged[merged['diff'] != 0]
lats = fluxpts['lat_t0'].to_numpy()
lons = fluxpts['lon_t0'].to_numpy()
s = m.Y21(lam, lats, lons)                                                   # assign Y21 values to these 'observation' points
s_noisy = m.add_noise(s, sigma=0.3)                                          # add noise
fig = m.plot_flux(lam, lats, lons, s_noisy, polygons=recon_polys)
plt.show()

In [ ]:
# flatten
ts, tlats, tlons = m.remove_zeros(m.make_ternary(s_noisy, -0.25, 0.25), lats, lons)
fig = m.plot_flux(lam, tlats, tlons, ts, polygons=recon_polys)
plt.show()

In [ ]:
# and invert
y = (ts == 1).astype(int)
df = m.invert_lam_logistic(tlats, tlons, y)
fig = m.plot_inversion(df)
plt.show()

In [ ]:
mask = df['beta'] > 0
best_idx = df.loc[mask, 'llr'].idxmax()                  
sbest = m.Y21(df.loc[best_idx, 'lambda'], tlats, tlons)  
m.plot_log_reg(sbest, y)                                 
plt.show()

In [ ]:
rlats, rlons = m.equisphere(len(ts))  
revals, revecs = m.eigendecomp(rlats, rlons)    
ref_power = np.sum(revals)                      

evals, evecs = m.eigendecomp(tlats, tlons)     
aniso = evals[0]/evals[1]                       
m.plot_resolving_pwr(evals, evecs, ref_power) 
plt.show()

In [ ]:
# Now with a global (eustatic) bias
s_biased = s + 0.33
s_biased_noisy = m.add_noise(s_biased, sigma=0.3)
fig = m.plot_flux(lam, lats, lons, s_biased_noisy, polygons=recon_polys)
plt.show()

In [ ]:
ts, tlats, tlons = m.remove_zeros(m.make_ternary(s_biased_noisy, -0.25, 0.25), lats, lons)
fig = m.plot_flux(lam, tlats, tlons, ts, polygons=recon_polys)
plt.show()

In [ ]:
y = (ts == 1).astype(int)
df = m.invert_lam_logistic(tlats, tlons, y)
fig = m.plot_inversion(df)
plt.show()

In [ ]:
mask = df['beta'] > 0
best_idx = df.loc[mask, 'llr'].idxmax()                  
sbest = m.Y21(df.loc[best_idx, 'lambda'], tlats, tlons)  
m.plot_log_reg(sbest, y)                                 
plt.show()

In [ ]:
evals, evecs = m.eigendecomp(tlats, tlons)     
aniso = evals[0]/evals[1]                       
m.plot_resolving_pwr(evals, evecs, ref_power) 
plt.show()

In [ ]:
# Finally, we can consider the statistical significance of the 'best-fit' by comparing with an ensemble of permutated inversions (where we randomly reassign the 'observed' signs)
clusters, n_clusters, noise = m.cluster(tlats, tlons, eps=3.0, min_samples=3)                # first we cluster the observations to account for spatial interdependence
perm_results = m.permutation_test_parallelized(ts, tlats, tlons, clusters, n=5000, n_jobs=5)   # then permute the observations and re-invert; do this n times
fig = m.plot_permutation_results(0, perm_results, n_clusters, noise)                         # collect, sort and plot the value of the best-fit result from each permutated model
plt.show()

In [ ]:
# We may now compare the 95% percentile of the permuted results with the best-fit we observed from our 'real' data
y = (ts == 1).astype(int)
df = m.invert_lam_logistic(tlats, tlons, y)
p95 = np.percentile(perm_results, 95)
fig = m.plot_inversion(df, p95)
plt.show()